<a href="https://colab.research.google.com/github/mahdisahad/WOA-Feature-Selection/blob/main/notebooks/03_Baseline_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_PATH = Path("/content/drive/MyDrive/WOA_Project")
DATA_PATH = PROJECT_PATH / "datasets" / "colon_dataset.csv"

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=["class"])
y = df["class"]

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Class distribution:")
print(y.value_counts())


Mounted at /content/drive
Dataset shape: (62, 2000)
Target shape: (62,)
Class distribution:
class
-1    40
 1    22
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


Training shape: (49, 2000)
Testing shape: (13, 2000)


In [ ]:
duplicate_columns = X_train.columns[
    X_train.T.duplicated()
].tolist()

print("Number of duplicate features:", len(duplicate_columns))

X_train = X_train.drop(columns=duplicate_columns)
X_test = X_test.drop(columns=duplicate_columns)

print("Training shape after duplicate removal:", X_train.shape)
print("Testing shape after duplicate removal:", X_test.shape)


Number of duplicate features: 9
Training shape after duplicate removal: (49, 1991)
Testing shape after duplicate removal: (13, 1991)


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    ]),

    "Linear SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="linear",
            class_weight="balanced"
        ))
    ]),

    "RBF SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="rbf",
            class_weight="balanced"
        ))
    ])
}

print("Baseline models defined successfully.")

Baseline models defined successfully.


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro"
}

cv_results = {}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    cv_results[name] = {
        "Accuracy": scores["test_accuracy"].mean(),
        "Balanced Accuracy": scores["test_balanced_accuracy"].mean(),
        "F1 Macro": scores["test_f1_macro"].mean()
    }

results_df = pd.DataFrame(cv_results).T

print(results_df.round(4))


                     Accuracy  Balanced Accuracy  F1 Macro
Logistic Regression    0.8578             0.8571    0.8434
Linear SVM             0.8578             0.8571    0.8434
RBF SVM                0.8378             0.8321    0.8177


In [ ]:
best_model_name = results_df["F1 Macro"].idxmax()

print("Best baseline model based on CV F1 Macro:")
print(best_model_name)

print("\nBest model results:")
print(results_df.loc[best_model_name])


Best baseline model based on CV F1 Macro:
Logistic Regression

Best model results:
Accuracy             0.857778
Balanced Accuracy    0.857143
F1 Macro             0.843382
Name: Logistic Regression, dtype: float64


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

test_results = {}

fitted_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    fitted_models[name] = model

    test_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precision Macro": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Recall Macro": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "F1 Macro": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        )
    }

test_results_df = pd.DataFrame(test_results).T

print(test_results_df.round(4))


                     Accuracy  Balanced Accuracy  Precision Macro  \
Logistic Regression    0.8462             0.8375           0.8375   
Linear SVM             0.8462             0.8375           0.8375   
RBF SVM                0.8462             0.8000           0.9000   

                     Recall Macro  F1 Macro  
Logistic Regression        0.8375    0.8375  
Linear SVM                 0.8375    0.8375  
RBF SVM                    0.8000    0.8194  


In [ ]:
best_model = fitted_models[best_model_name]

y_pred_best = best_model.predict(X_test)

print("Best model:", best_model_name)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))


Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

          -1       0.88      0.88      0.88         8
           1       0.80      0.80      0.80         5

    accuracy                           0.85        13
   macro avg       0.84      0.84      0.84        13
weighted avg       0.85      0.85      0.85        13


Confusion Matrix:
[[7 1]
 [1 4]]


In [ ]:
RESULTS_PATH = PROJECT_PATH / "results"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    RESULTS_PATH / "baseline_cv_results.csv"
)

test_results_df.to_csv(
    RESULTS_PATH / "baseline_test_results.csv"
)

print("Baseline results saved successfully")


Baseline results saved successfully
